# Multi-task model training on Colab (Phase 3d)

Generated from `xBD_damage_to_carbon_cost.ipynb` (Section 3c-ii) by `tools/make_colab_notebook.py`. The model, loss, data pipeline and training loop are **the same cells** as in the main notebook; only this setup cell differs.

**Before running:** Runtime → Change runtime type → **A100 GPU** (or L4) and **High-RAM**. Then Runtime → Run all, and approve the Google Drive prompt.

**It needs** three files somewhere in `My Drive/ENGG2112_Project/`: `mt_tiles_train.zip`, `mt_tiles_tier3.zip`, `mt_tiles_test.zip`. They are unpacked onto Colab's own disk (a few minutes).

**It prints one line per epoch** (validation damage macro F1 and building IoU), so progress is visible. The best epoch's weights are kept.

**It produces** four files in the same Drive folder, starting with `mt_unet_effnet_`: `..._history.csv`, `..._test_predictions.csv`, `..._test_building_map.csv` and `....weights.h5`. Download all four into `xbd_project/outputs/` on the laptop.

In [ ]:
import time, zipfile
from pathlib import Path
import numpy as np, pandas as pd
from PIL import Image
import keras
from keras import layers
from sklearn.metrics import f1_score

from google.colab import drive
drive.mount("/content/drive")
DRIVE_FOLDER = Path("/content/drive/MyDrive/ENGG2112_Project")
zips = {z.stem.replace("mt_tiles_", ""): z for z in DRIVE_FOLDER.rglob("mt_tiles_*.zip")}
missing = [s for s in ("train", "tier3", "test") if s not in zips]
assert not missing, f"not found under {DRIVE_FOLDER}: {[f'mt_tiles_{s}.zip' for s in missing]}"
OUTPUT_DIR = zips["test"].parent          # results are written next to the zips
TILE_DIR = Path("/content/mt_tiles")
for split, z in zips.items():
    if not (TILE_DIR / split / "tiles.csv").exists():
        print("unpacking", z.name, round(z.stat().st_size / 1e9, 2), "GB"); zipfile.ZipFile(z).extractall(TILE_DIR)
print("Results will be written to:", OUTPUT_DIR)

import tensorflow as tf
gpus = tf.config.list_physical_devices("GPU")
print("GPU:", gpus or "NONE - this model needs a GPU")
MIXED_PRECISION = True   # set to False if training stops with a precision error; slower but always works
if gpus and MIXED_PRECISION:
    keras.mixed_precision.set_global_policy("mixed_float16")   # about twice as fast on A100/L4/T4; outputs stay float32

DAMAGE_CLASSES = ["no-damage", "minor-damage", "major-damage", "destroyed"]  # ordinal order
p3_histories, model_predictions = [], {}

In [ ]:
import tensorflow as tf

MT_CROP = 512                      # training window size (pixels); prediction uses the full 1024 tile
MT_BATCH = 8
MT_DECODER_CHANNELS = [256, 128, 64, 32, 16]
MT_SKIP_LAYERS = ["block2a_expand_activation", "block3a_expand_activation", "block4a_expand_activation",
                  "block6a_expand_activation", "top_activation"]          # EfficientNetB0 features at 1/2 ... 1/32 scale

MT_EXPERIMENTS = pd.DataFrame([
    ("mt_unet_effnet", "train+tier3", 30, 500, 3e-4, 0),
], columns=["name", "train_on", "epochs", "steps_per_epoch", "lr", "seed"])


def conv_block(x, channels):
    for _ in range(2):
        x = layers.Conv2D(channels, 3, padding="same", use_bias=False)(x)
        x = layers.BatchNormalization()(x)
        x = layers.ReLU()(x)
    return x


def build_multitask_unet():
    """Siamese U-Net: before/after tiles -> per-pixel [building score, 4 damage scores]."""
    before = keras.Input((None, None, 3), name="before")
    after = keras.Input((None, None, 3), name="after")
    backbone = keras.applications.EfficientNetB0(include_top=False, weights="imagenet", input_shape=(None, None, 3))
    encoder = keras.Model(backbone.input, [backbone.get_layer(n).output for n in MT_SKIP_LAYERS], name="shared_encoder")
    f_before, f_after = encoder(before), encoder(after)                  # ONE encoder, used twice
    joined = [layers.Concatenate()([a, b, layers.Subtract()([b, a])]) for a, b in zip(f_before, f_after)]
    x = conv_block(joined[-1], MT_DECODER_CHANNELS[0])                   # start at 1/32 scale
    for skip, channels in zip(reversed(joined[:-1]), MT_DECODER_CHANNELS[1:]):
        x = layers.UpSampling2D(interpolation="bilinear")(x)             # double the size ...
        x = conv_block(layers.Concatenate()([x, skip]), channels)        # ... and merge that scale's features
    x = layers.UpSampling2D(interpolation="bilinear")(x)                 # back to full size
    maps = layers.Conv2D(5, 1, dtype="float32", name="maps")(x)          # channel 0: building, 1-4: damage
    return keras.Model([before, after], maps, name="multitask_unet")


def multitask_loss(target, maps):
    """Building map: cross-entropy + Dice. Damage map: cross-entropy on labelled building pixels only."""
    target = tf.cast(tf.reshape(target, tf.shape(maps)[:3]), tf.int32)
    is_building = tf.cast(target > 0, tf.float32)
    building_score = maps[..., 0]
    bce = tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(labels=is_building, logits=building_score))
    p = tf.sigmoid(building_score)
    dice = 1.0 - (2.0 * tf.reduce_sum(p * is_building) + 1.0) / (tf.reduce_sum(p) + tf.reduce_sum(is_building) + 1.0)
    labelled = tf.cast((target >= 1) & (target <= 4), tf.float32)
    damage_ce = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=tf.clip_by_value(target - 1, 0, 3), logits=maps[..., 1:])
    damage = tf.reduce_sum(damage_ce * labelled) / tf.maximum(tf.reduce_sum(labelled), 1.0)
    return bce + dice + damage


def read_tile(tile_path_prefix):
    """Before, after, target and instance images of one tile (as tensors)."""
    jpg = lambda s: tf.io.decode_jpeg(tf.io.read_file(tile_path_prefix + s), channels=3)
    png = lambda s, dtype: tf.io.decode_png(tf.io.read_file(tile_path_prefix + s), channels=1, dtype=dtype)
    return jpg("_pre.jpg"), jpg("_post.jpg"), png("_target.png", tf.uint8)


def training_windows(prefixes, seed):
    """Endless stream of random 512 x 512 windows from random training tiles, with flips and
    90-degree rotations applied identically to before, after and target."""
    def window(prefix):
        before, after, target = read_tile(prefix)
        stack = tf.concat([before, after, target], axis=-1)                    # 7 channels, cropped together
        stack = tf.image.random_crop(stack, (MT_CROP, MT_CROP, 7))
        stack = tf.image.random_flip_left_right(stack)
        stack = tf.image.rot90(stack, k=tf.random.uniform((), 0, 4, dtype=tf.int32))
        return (tf.cast(stack[..., :3], tf.float32), tf.cast(stack[..., 3:6], tf.float32)), stack[..., 6]
    return (tf.data.Dataset.from_tensor_slices(prefixes).shuffle(len(prefixes), seed=seed).repeat()
            .map(window, num_parallel_calls=tf.data.AUTOTUNE).batch(MT_BATCH).prefetch(tf.data.AUTOTUNE))


def predict_tiles(model, tile_dir, tile_names, instances):
    """Run the model on whole tiles. Returns per-building damage predictions (majority vote of the
    damage map inside each ground-truth building) and per-tile building-map pixel counts."""
    building_rows, pixel_rows = [], []
    by_tile = instances.groupby("tile")
    for tile in tile_names:
        before, after, target = (t.numpy() for t in read_tile(f"{tile_dir}/{tile}"))
        maps = model.predict_on_batch((before[None].astype(np.float32), after[None].astype(np.float32)))[0]
        pred_building, true_building = maps[..., 0] > 0, target[..., 0] > 0
        pixel_rows.append({"tile": tile, "tp": int((pred_building & true_building).sum()),
                           "fp": int((pred_building & ~true_building).sum()), "fn": int((~pred_building & true_building).sum())})
        if tile not in by_tile.groups:
            continue
        inst = np.asarray(Image.open(f"{tile_dir}/{tile}_instances.png"), dtype=np.int64)
        votes = np.bincount((inst * 4 + maps[..., 1:].argmax(axis=-1)).ravel(), minlength=(inst.max() + 1) * 4).reshape(-1, 4)
        for row in by_tile.get_group(tile).itertuples():
            if row.damage_class not in DAMAGE_CLASSES:
                continue
            covered = row.instance < len(votes) and votes[row.instance].sum() > 0   # False only if fully drawn over by a neighbour
            building_rows.append({"file": f"{tile}_post_disaster", "uid": row.uid, "disaster": tile.split("_")[0],
                                  "true_class": row.damage_class,
                                  "pred_class": DAMAGE_CLASSES[int(votes[row.instance].argmax())] if covered else "no-damage",
                                  "no_pixels": not covered})
    return pd.DataFrame(building_rows), pd.DataFrame(pixel_rows)

In [ ]:
MT_TRAIN = True                  # on Colab: train every multi-task experiment with no result yet
MT_TILE_DIR = TILE_DIR                 # unpacked on Colab's local disk by the first cell


def run_multitask(exp):
    """Train one row of MT_EXPERIMENTS (or load it from cache)."""
    tag = f"{exp.name}_{exp.epochs}ep_{exp.steps_per_epoch}steps"
    paths = {k: OUTPUT_DIR / f"{tag}{s}" for k, s in [("weights", ".weights.h5"), ("history", "_history.csv"),
                                                      ("preds", "_test_predictions.csv"), ("pixels", "_test_building_map.csv")]}
    if all(paths[k].exists() for k in ("history", "preds", "pixels")):
        return pd.read_csv(paths["history"]), pd.read_csv(paths["preds"]), pd.read_csv(paths["pixels"])
    if not MT_TRAIN:
        return None

    index = pd.concat(pd.read_csv(MT_TILE_DIR / s / "tiles.csv") for s in ("train", "tier3", "test"))
    instances = pd.concat(pd.read_csv(MT_TILE_DIR / s / "instances.csv") for s in ("train", "tier3", "test"))
    prefix = lambda r: str(MT_TILE_DIR / r.split / r.tile)
    train_rows = index[(index["subset"] == "train") & index["split"].isin(exp.train_on.split("+"))]
    val_rows, test_rows = index[index["subset"] == "validation"], index[index["subset"] == "test"]

    keras.utils.set_random_seed(exp.seed)
    model = build_multitask_unet()
    schedule = keras.optimizers.schedules.CosineDecay(exp.lr, decay_steps=exp.epochs * exp.steps_per_epoch)
    # jit_compile=False: TensorFlow's XLA compiler rejects the mix of 16- and 32-bit numbers that
    # mixed precision creates in the bilinear up-sampling gradient (error seen on Colab, 2026-10-15)
    model.compile(optimizer=keras.optimizers.Adam(schedule), loss=multitask_loss, jit_compile=False)
    windows = training_windows([prefix(r) for r in train_rows.itertuples()], exp.seed)

    history, best, start = [], -1.0, time.time()
    for epoch in range(1, exp.epochs + 1):
        fit = model.fit(windows, steps_per_epoch=exp.steps_per_epoch, epochs=1, verbose=0)
        val_buildings, val_pixels = predict_tiles(model, str(MT_TILE_DIR / "train"), val_rows["tile"], instances)
        val_f1 = f1_score(val_buildings["true_class"], val_buildings["pred_class"], labels=DAMAGE_CLASSES, average="macro")
        tp, fp, fn = val_pixels[["tp", "fp", "fn"]].sum()
        history.append({"experiment": exp.name, "epoch": epoch, "train_loss": fit.history["loss"][0],
                        "val_macro_f1": val_f1, "val_building_iou": tp / (tp + fp + fn), "minutes": (time.time() - start) / 60})
        print(f"epoch {epoch}: loss {history[-1]['train_loss']:.3f}  val damage macro F1 {val_f1:.3f}  val building IoU {history[-1]['val_building_iou']:.3f}")
        if val_f1 > best:
            best = val_f1
            model.save_weights(paths["weights"])
    model.load_weights(paths["weights"])
    test_buildings, test_pixels = predict_tiles(model, str(MT_TILE_DIR / "test"), test_rows["tile"], instances)
    history = pd.DataFrame(history)
    history.to_csv(paths["history"], index=False); test_buildings.to_csv(paths["preds"], index=False); test_pixels.to_csv(paths["pixels"], index=False)
    return history, test_buildings, test_pixels


mt_results = {}
for exp in MT_EXPERIMENTS.itertuples(index=False):
    result = run_multitask(exp)
    if result is None:
        print(f"{exp.name}: not trained yet (no cached result and MT_TRAIN is False)")
        continue
    mt_results[exp.name] = result
    p3_histories.append(result[0]); model_predictions[exp.name] = result[1]   # joins the selection and Section 7b
    best = result[0].loc[result[0]["val_macro_f1"].idxmax()]
    print(f"{exp.name}: best validation macro F1 {best['val_macro_f1']:.3f} at epoch {int(best['epoch'])} "
          f"({result[0]['minutes'].max():.0f} min of training)")

In [ ]:
for name, (history, buildings, pixels) in mt_results.items():
    per_class = f1_score(buildings["true_class"], buildings["pred_class"], labels=DAMAGE_CLASSES, average=None)
    tp, fp, fn = pixels[["tp", "fp", "fn"]].sum()
    print(f"{name}: test damage macro F1 {per_class.mean():.3f} "
          f"(per class {np.round(per_class, 3).tolist()}), test building IoU {tp / (tp + fp + fn):.3f}")
print("Files to download into xbd_project/outputs/:")
for f in sorted(OUTPUT_DIR.glob("mt_unet_*")): print("  ", f.name)